# 1.3 : Brand standards configuration example

**Acceptance criterion:** When the configuration is loaded, all MVP criteria parse successfully and expose the fields required by the scoring/evaluation pipeline.

This example loads the two files in `config/`, validates their common structure, checks unique IDs and the six locked MVP entries, and displays the evaluator interface. The saved output was executed using the latest configuration and schema supplied in this conversation. Rerun all cells against your repository files before submission.

In [4]:
import json
from pathlib import Path
import os
from collections import Counter
from jsonschema import Draft202012Validator

# Notebook runs from examples; config folder is one level above.
repo_root = Path.cwd().parent
config_path = os.path.join(repo_root, "config", "brand_standards.json")
schema_path = os.path.join(repo_root, "config", "brand_standards.schema.json")

standards = json.loads(Path(config_path).read_text(encoding="utf-8"))
schema = json.loads(Path(schema_path).read_text(encoding="utf-8"))
print("Loaded: config/brand_standards.json")
print("Loaded: config/brand_standards.schema.json")
print("Rubric:", standards["rubric_id"])

Loaded: config/brand_standards.json
Loaded: config/brand_standards.schema.json
Rubric: default-v1-draft


## Validate against JSON Schema

The schema checks required fields, types, method options, weight/flag ranges and palette formatting. Criterion-specific `params` remain flexible.

In [2]:
Draft202012Validator.check_schema(schema)
validator = Draft202012Validator(schema)
errors = sorted(validator.iter_errors(standards), key=lambda e: str(list(e.absolute_path)))
if errors:
    details = "\n".join(f"{list(e.absolute_path)}: {e.message}" for e in errors)
    raise ValueError("Configuration validation failed:\n" + details)
print("PASS: schema is valid and configuration satisfies it.")

PASS: schema is valid and configuration satisfies it.


## Verify the locked MVP and unique IDs

The reusable schema permits future criterion IDs. This demonstration separately checks that the current six locked criteria are included; additional criteria are allowed.

In [5]:
required_mvp_ids = {
    "exposure_brightness", "contrast", "color_palette_fit",
    "framing_subject_position", "sharpness_subject_emphasis",
    "tone_approachability",
}

criteria = standards["criteria"]
ids = [c["criterion_id"] for c in criteria]
duplicates = [cid for cid, count in Counter(ids).items() if count > 1]
if duplicates:
    raise ValueError(f"Duplicate criterion IDs: {duplicates}")

missing = required_mvp_ids - set(ids)
if missing:
    raise ValueError(f"Missing MVP criteria: {sorted(missing)}")

print("PASS: criterion IDs are unique.")
print("PASS: all six locked MVP criteria are present.")
print("Total criteria:", len(criteria))

PASS: criterion IDs are unique.
PASS: all six locked MVP criteria are present.
Total criteria: 6


## Inspect the pipeline fields

In [6]:
for criterion in criteria:
    print(f"{criterion['criterion_id']} | method={criterion['method']} | weight={criterion['weight']}")
    print("  Required outputs:", ", ".join(criterion["required_measurements"]) or "none")
    print("  Parameter keys:", ", ".join(criterion["params"]))
    print("  Definition/evidence fields: present")

exposure_brightness | method=cv | weight=None
  Required outputs: mean_luminance, near_black_fraction, near_white_fraction
  Parameter keys: luminance_scale, near_black_max, near_white_min, mean_luminance_min, mean_luminance_max, near_black_fraction_max, near_white_fraction_max, threshold_status
  Definition/evidence fields: present
contrast | method=cv | weight=None
  Required outputs: luminance_std, p5_p95_range, near_black_fraction, near_white_fraction
  Parameter keys: luminance_std_min, luminance_std_max, p5_p95_range_min, p5_p95_range_max, near_black_max, near_white_min, near_black_fraction_max, near_white_fraction_max, threshold_status
  Definition/evidence fields: present
color_palette_fit | method=cv | weight=None
  Required outputs: dominant_colors, palette_distance
  Parameter keys: n_clusters, random_seed, color_space, distance_metric, palette_distance_aggregation, palette_distance_max, threshold_status
  Definition/evidence fields: present
framing_subject_position | method

## Acceptance result and limits

Successful configuration loading is separate from scoring readiness. Still need to finalize weights, calibrated thresholds and CV score mappings. This example does not validate every parameter value, or verify API/database integration.

In [7]:
print("ACCEPTANCE CHECK PASSED: all MVP criteria load and expose the agreed configuration fields.")
print("Configuration status:", standards["metadata"]["status"])
print("Scoring ready:", standards["metadata"]["scoring_ready"])
print("Weights status:", standards["metadata"]["weights_status"])
print("Flag threshold:", standards["flag_threshold"])
print("Palette status:", standards["metadata"]["palette_status"])

ACCEPTANCE CHECK PASSED: all MVP criteria load and expose the agreed configuration fields.
Configuration status: draft
Scoring ready: False
Weights status: unset
Flag threshold: None
Palette status: placeholder_not_brand_confirmed
